In [10]:
import os
os.environ["KERAS_BACKEND"] = "torch" 
from functools import partial
from molexpress import layers
from molexpress.datasets import featurizers
from molexpress.datasets import encoders
from molexpress.ops.chem_ops import get_molecule
import torch
import pandas as pd 
from tqdm import tqdm


class GraphNeuralNetwork(torch.nn.Module):
    
    def __init__(self, dim):
        super().__init__()
        self.gcn1 = layers.GINConv(dim)
        self.gcn2 = layers.GINConv(dim)
        self.gcn3 = layers.GINConv(dim)
        self.gcn4 = layers.GINConv(dim)
        
    def forward(self, x):
        x = self.gcn1(x)
        x = self.gcn2(x)
        x = self.gcn3(x)
        x = self.gcn4(x)
        return x


class NodePrediction(torch.nn.Module):
    
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear1 = torch.nn.Linear(input_dim, input_dim) 
        self.linear2 = torch.nn.Linear(input_dim, output_dim) 
        
    def forward(self, x):
        x = self.linear1(x['node_state'])
        x = torch.nn.functional.relu(x,inplace=False)
        x = self.linear2(x)
        return x


class EdgePrediction(torch.nn.Module):
    
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear1 = torch.nn.Linear(input_dim, input_dim) 
        self.linear2 = torch.nn.Linear(input_dim, output_dim)
        self.gather_incident = layers.GatherIncident()
        
    def forward(self, x):
        x = self.gather_incident(x) # We do not use edge states but incident node states.
        x = self.linear1(x)
        x = torch.nn.functional.relu(x,inplace=False)
        x = self.linear2(x)
        return x
    

atom_featurizers = [
    featurizers.AtomType(vocab={'C', 'N', 'O'}),
    featurizers.Hybridization(),
]

bond_featurizers = [
    featurizers.BondType(),
    featurizers.Conjugated()
]

peptide_graph_encoder = encoders.PeptideGraphEncoder(
    atom_featurizers=atom_featurizers, 
    bond_featurizers=bond_featurizers,
    self_loops=False, # self_loops True adds one feature dim to edge state
    supports_masking=True, # supports_masking True adds one feature dim to node and edge state
)



In [11]:
class Dataset(torch.utils.data.Dataset):
    
    def __init__(self, x):
        self.x = x

    def __len__(self):
        return len(self.x)
        
    def __getitem__(self, index):
        graph = peptide_graph_encoder(self.x[index])
        return graph
s


In [12]:
data = pd.read_csv("/home/harikrishnan/molexpress-main/molexpress/pretraining/filtered_pubchem.txt",names=["smiles"])

dataset = data["smiles"].apply(lambda x: [x]).to_list()[400000:]
# print(len(dataset))




In [13]:
torch_dataset = Dataset(dataset)

partial_collate_fn = partial(
    peptide_graph_encoder.masked_collate_fn, node_masking_rate=0.3, edge_masking_rate=0.3)

dataset = torch.utils.data.DataLoader(
    torch_dataset, batch_size=1024, collate_fn=partial_collate_fn,num_workers= 6)

In [ ]:
for ind, batch in enumerate(dataset):
    

